In [ ]:
"""
6-Factor Fundamental Analysis Toolkit
──────────────────────────────────────
  1. Leverage / Debt Structure
  2. Net Cash Position
  3. Operating Cash Flow
  4. Capital Allocation Efficiency
  5. Sustainable Earnings / Core Profit
  6. Tangible Book Value

Usage: change tickers at the bottom and run.
Requires: pip install yfinance pandas numpy
"""

import json, math, sys, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Data Model
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class FinancialData:
    year: int
    # Income Statement
    revenue: float = 0.0
    cost_of_revenue: float = 0.0
    gross_profit: float = 0.0
    operating_income: float = 0.0
    net_income: float = 0.0
    ebitda: float = 0.0
    interest_expense: float = 0.0
    non_recurring_income: float = 0.0
    # Balance Sheet
    total_assets: float = 0.0
    total_liabilities: float = 0.0
    total_equity: float = 0.0
    goodwill: float = 0.0
    intangible_assets: float = 0.0
    cash_and_equivalents: float = 0.0
    short_term_investments: float = 0.0
    total_debt: float = 0.0
    short_term_debt: float = 0.0
    long_term_debt: float = 0.0
    current_assets: float = 0.0
    current_liabilities: float = 0.0
    fixed_assets: float = 0.0
    # Cash Flow
    operating_cash_flow: float = 0.0
    capex: float = 0.0
    free_cash_flow: float = 0.0
    dividends_paid: float = 0.0
    share_buyback: float = 0.0
    # Market
    market_cap: float = 0.0
    shares_outstanding: float = 0.0
    stock_price: float = 0.0


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  6-Factor Analyzer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SixFactorAnalyzer:

    WEIGHTS = {
        "leverage": 1/6,
        "net_cash": 1/6,
        "cash_flow": 1/6,
        "capital_allocation": 1/6,
        "earnings_quality": 1/6,
        "tangible_book": 1/6,
    }

    def __init__(self, data: list[FinancialData], ticker: str = "",
                 ttm: FinancialData = None):
        self.data = sorted(data, key=lambda d: d.year)
        self.latest = self.data[-1]
        self.ttm = ttm
        self.ticker = ticker
        self.results: dict = {}

    # ── 1. Leverage / Debt Structure ──
    def analyze_leverage(self) -> dict:
        d = self.latest
        m = {}
        m["Debt/Assets"] = d.total_liabilities / d.total_assets if d.total_assets else 0
        m["Debt/Equity"] = d.total_liabilities / d.total_equity if d.total_equity else 0
        m["Interest-Bearing Debt Ratio"] = d.total_debt / d.total_assets if d.total_assets else 0
        m["Short-Term Debt %"] = d.short_term_debt / d.total_debt if d.total_debt else 0
        m["Current Ratio"] = d.current_assets / d.current_liabilities if d.current_liabilities else 0
        m["Interest Coverage"] = d.ebitda / d.interest_expense if d.interest_expense > 0 else float("inf")

        score = 50
        dar = m["Debt/Assets"]
        if dar < 0.3: score += 20
        elif dar < 0.5: score += 10
        elif dar > 0.7: score -= 20
        cr = m["Current Ratio"]
        if cr > 2: score += 10
        elif cr > 1.5: score += 5
        elif cr < 1: score -= 15
        ic = m["Interest Coverage"]
        if ic == float("inf") or ic > 10: score += 10
        elif ic < 3: score -= 15
        st = m["Short-Term Debt %"]
        if st > 0.5 and d.total_debt > 0: score -= 10

        m["Score"] = max(0, min(100, score))
        self.results["leverage"] = m
        return m

    # ── 2. Net Cash Position ──
    def analyze_net_cash(self) -> dict:
        d = self.latest
        m = {}
        net = d.cash_and_equivalents + d.short_term_investments - d.total_debt
        m["Cash + ST Investments"] = d.cash_and_equivalents + d.short_term_investments
        m["Total Debt"] = d.total_debt
        m["Net Cash"] = net
        m["Net Cash / Assets"] = net / d.total_assets if d.total_assets else 0
        m["Net Cash / Market Cap"] = net / d.market_cap if d.market_cap else 0
        m["Positive"] = net > 0

        # Trend over years
        trend = []
        for yr in self.data:
            nc = yr.cash_and_equivalents + yr.short_term_investments - yr.total_debt
            trend.append({"year": yr.year, "net_cash": nc})
        m["Trend"] = trend

        score = 50
        if net > 0: score += 25
        elif net > -d.total_assets * 0.1: score += 5
        else: score -= 15
        ratio = m["Net Cash / Assets"]
        if ratio > 0.2: score += 15
        elif ratio > 0.1: score += 5
        elif ratio < -0.3: score -= 10
        # Improving or worsening?
        if len(trend) >= 2 and trend[-1]["net_cash"] > trend[0]["net_cash"]:
            score += 5

        m["Score"] = max(0, min(100, score))
        self.results["net_cash"] = m
        return m

    # ── 3. Operating Cash Flow ──
    def analyze_cash_flow(self) -> dict:
        d = self.latest
        m = {}
        m["Operating CF"] = d.operating_cash_flow
        m["Net Income"] = d.net_income
        m["OCF / Net Income"] = d.operating_cash_flow / d.net_income if d.net_income > 0 else 0
        fcf = d.free_cash_flow if d.free_cash_flow else d.operating_cash_flow - d.capex
        m["Free Cash Flow"] = fcf
        m["FCF Margin"] = fcf / d.revenue if d.revenue else 0
        m["Capex / OCF"] = d.capex / d.operating_cash_flow if d.operating_cash_flow > 0 else 0

        # Cumulative check
        tot_ocf = sum(y.operating_cash_flow for y in self.data)
        tot_ni = sum(y.net_income for y in self.data)
        m["Cumulative OCF/NI"] = tot_ocf / tot_ni if tot_ni > 0 else 0

        # OCF trend
        m["OCF Trend"] = [{"year": y.year, "ocf": y.operating_cash_flow} for y in self.data]

        score = 50
        o = m["OCF / Net Income"]
        if o > 1.2: score += 20
        elif o > 0.8: score += 10
        elif o < 0.5: score -= 20
        if m["FCF Margin"] > 0.15: score += 10
        elif m["FCF Margin"] > 0.05: score += 5
        elif m["FCF Margin"] < 0: score -= 15
        if m["Cumulative OCF/NI"] > 1.0: score += 10
        elif m["Cumulative OCF/NI"] < 0.7: score -= 10

        m["Score"] = max(0, min(100, score))
        self.results["cash_flow"] = m
        return m

    # ── 4. Capital Allocation Efficiency ──
    def analyze_capital_allocation(self) -> dict:
        d = self.latest
        m = {}
        m["ROE"] = d.net_income / d.total_equity if d.total_equity else 0
        m["ROA"] = d.net_income / d.total_assets if d.total_assets else 0
        ic = d.total_equity + d.total_debt - d.cash_and_equivalents
        nopat = d.operating_income * 0.75
        m["ROIC"] = nopat / ic if ic > 0 else 0

        # Shareholder returns
        total_return = d.dividends_paid + d.share_buyback
        m["Dividends"] = d.dividends_paid
        m["Buybacks"] = d.share_buyback
        m["Total Return to Shareholders"] = total_return
        m["Payout Ratio"] = total_return / d.net_income if d.net_income > 0 else 0

        # Capex discipline
        fcf = d.free_cash_flow if d.free_cash_flow else d.operating_cash_flow - d.capex
        m["FCF after Returns"] = fcf - total_return
        m["Reinvestment Rate"] = d.capex / d.operating_cash_flow if d.operating_cash_flow > 0 else 0

        score = 50
        if m["ROIC"] > 0.20: score += 20
        elif m["ROIC"] > 0.12: score += 10
        elif m["ROIC"] > 0.06: score += 5
        elif m["ROIC"] < 0: score -= 15
        if m["ROE"] > 0.15: score += 10
        elif m["ROE"] < 0.05: score -= 10
        if m["Payout Ratio"] > 0.3 and m["Payout Ratio"] < 0.8: score += 5
        if m["Reinvestment Rate"] > 0.8: score -= 10

        m["Score"] = max(0, min(100, score))
        self.results["capital_allocation"] = m
        return m

    # ── 5. Sustainable Earnings / Core Profit ──
    def analyze_earnings_quality(self) -> dict:
        d = self.latest
        m = {}
        core = d.net_income - d.non_recurring_income
        m["Net Income"] = d.net_income
        m["Non-Recurring Items"] = d.non_recurring_income
        m["Core Profit"] = core
        m["Core Profit %"] = core / d.net_income if d.net_income else 0

        # OCF vs NI as quality check
        m["OCF / Net Income"] = d.operating_cash_flow / d.net_income if d.net_income > 0 else 0

        # Earnings stability (std of NI margins across years)
        margins = []
        for y in self.data:
            if y.revenue > 0:
                margins.append(y.net_income / y.revenue)
        if len(margins) >= 2:
            m["Margin Stability (CV)"] = float(np.std(margins) / abs(np.mean(margins))) if np.mean(margins) != 0 else None
        else:
            m["Margin Stability (CV)"] = None

        # NI trend
        m["NI Trend"] = [{"year": y.year, "ni": y.net_income} for y in self.data]

        score = 50
        cp = m["Core Profit %"]
        if cp > 0.9: score += 20
        elif cp > 0.7: score += 10
        elif cp < 0.5: score -= 20
        o = m["OCF / Net Income"]
        if o > 1.0: score += 10
        elif o < 0.6: score -= 10
        stab = m.get("Margin Stability (CV)")
        if stab is not None:
            if stab < 0.3: score += 10
            elif stab > 1.0: score -= 10

        m["Score"] = max(0, min(100, score))
        self.results["earnings_quality"] = m
        return m

    # ── 6. Tangible Book Value ──
    def analyze_tangible_book(self) -> dict:
        d = self.latest
        m = {}
        m["Total Equity"] = d.total_equity
        m["Goodwill"] = d.goodwill
        m["Intangible Assets"] = d.intangible_assets
        tangible = d.total_equity - d.goodwill - d.intangible_assets
        m["Tangible Book Value"] = tangible
        m["Goodwill / Equity"] = d.goodwill / d.total_equity if d.total_equity else 0
        m["Intangibles / Equity"] = (d.goodwill + d.intangible_assets) / d.total_equity if d.total_equity else 0
        m["Tangible BV / Share"] = tangible / d.shares_outstanding if d.shares_outstanding else 0
        m["Price / Tangible BV"] = d.market_cap / tangible if tangible > 0 else None
        m["Tangible Positive"] = tangible > 0

        score = 50
        gw_eq = m["Goodwill / Equity"]
        if gw_eq < 0.05: score += 20
        elif gw_eq < 0.20: score += 10
        elif gw_eq > 0.50: score -= 20
        elif gw_eq > 0.30: score -= 10
        if tangible > 0: score += 10
        else: score -= 20
        int_eq = m["Intangibles / Equity"]
        if int_eq > 0.6: score -= 10

        m["Score"] = max(0, min(100, score))
        self.results["tangible_book"] = m
        return m

    # ── Run All ──
    def run_full_analysis(self):
        self.analyze_leverage()
        self.analyze_net_cash()
        self.analyze_cash_flow()
        self.analyze_capital_allocation()
        self.analyze_earnings_quality()
        self.analyze_tangible_book()

        total = sum(
            self.results[k].get("Score", 50) * w
            for k, w in self.WEIGHTS.items() if k in self.results
        )
        self.results["total_score"] = round(total, 1)

        ts = self.results["total_score"]
        if ts >= 80: self.results["grade"] = "A  Excellent"
        elif ts >= 65: self.results["grade"] = "B  Good"
        elif ts >= 50: self.results["grade"] = "C  Average"
        elif ts >= 35: self.results["grade"] = "D  Below Average"
        else: self.results["grade"] = "F  Poor"

        self.results["risks"] = self._risks()
        self.results["highlights"] = self._highlights()
        return self

    def _risks(self) -> list[str]:
        r = []
        lv = self.results.get("leverage", {})
        nc = self.results.get("net_cash", {})
        cf = self.results.get("cash_flow", {})
        ca = self.results.get("capital_allocation", {})
        eq = self.results.get("earnings_quality", {})
        tb = self.results.get("tangible_book", {})

        if lv.get("Debt/Assets", 0) > 0.6:
            r.append(f"High leverage (D/A {lv['Debt/Assets']:.0%})")
        if lv.get("Interest Coverage", 999) < 3:
            r.append(f"Weak interest coverage ({lv['Interest Coverage']:.1f}x)")
        if not nc.get("Positive", True):
            r.append(f"Negative net cash ({nc['Net Cash']:,.0f})")
        if cf.get("OCF / Net Income", 1) < 0.6:
            r.append(f"OCF doesn't support earnings ({cf['OCF / Net Income']:.2f}x)")
        if ca.get("ROIC", 0) < 0.05:
            r.append(f"Low ROIC ({ca['ROIC']:.1%})")
        if eq.get("Core Profit %", 1) < 0.6:
            r.append(f"Low core profit ({eq['Core Profit %']:.0%} of NI)")
        if tb.get("Goodwill / Equity", 0) > 0.4:
            r.append(f"Heavy goodwill ({tb['Goodwill / Equity']:.0%} of equity)")
        if not tb.get("Tangible Positive", True):
            r.append("Negative tangible book value")
        return r

    def _highlights(self) -> list[str]:
        h = []
        lv = self.results.get("leverage", {})
        nc = self.results.get("net_cash", {})
        cf = self.results.get("cash_flow", {})
        ca = self.results.get("capital_allocation", {})
        eq = self.results.get("earnings_quality", {})
        tb = self.results.get("tangible_book", {})

        if lv.get("Debt/Assets", 1) < 0.3:
            h.append(f"Low leverage (D/A {lv['Debt/Assets']:.0%})")
        if nc.get("Positive"):
            h.append(f"Net cash {nc['Net Cash']:,.0f}")
        if cf.get("OCF / Net Income", 0) > 1.0:
            h.append(f"Strong cash conversion ({cf['OCF / Net Income']:.2f}x)")
        if ca.get("ROIC", 0) > 0.15:
            h.append(f"High ROIC ({ca['ROIC']:.1%})")
        if eq.get("Core Profit %", 0) > 0.9:
            h.append("Clean earnings (>90% core)")
        if tb.get("Goodwill / Equity", 1) < 0.05:
            h.append("Minimal goodwill")
        return h

    # ── Print Report ──
    def print_report(self):
        if not self.results:
            self.run_full_analysis()
        r = self.results
        d = self.latest

        def f(v, s="pct"):
            if v is None: return "N/A"
            if isinstance(v, float) and math.isinf(v): return "INF"
            if s == "pct": return f"{v:.1%}"
            if s == "x": return f"{v:.2f}x"
            if s == "n": return f"{v:,.0f}"
            return str(v)

        W = 60
        name = self.ticker.upper() or "Company"
        print(f"\n{'=' * W}")
        print(f"  {name}  |  FY{d.year}  |  Score: {r['total_score']}/100  [{r['grade']}]")
        print(f"{'=' * W}")

        # 1. Leverage
        lv = r["leverage"]
        print(f"\n  1. LEVERAGE ({lv['Score']}/100)")
        print(f"     Debt/Assets        {f(lv['Debt/Assets']):<10}  Debt/Equity       {f(lv['Debt/Equity'], 'x')}")
        print(f"     Current Ratio      {f(lv['Current Ratio'], 'x'):<10}  Interest Coverage {f(lv['Interest Coverage'], 'x')}")
        print(f"     ST Debt %          {f(lv['Short-Term Debt %'])}")

        # 2. Net Cash
        nc = r["net_cash"]
        print(f"\n  2. NET CASH ({nc['Score']}/100)")
        print(f"     Cash + ST Inv      {f(nc['Cash + ST Investments'], 'n')}")
        print(f"     Total Debt         {f(nc['Total Debt'], 'n')}")
        print(f"     Net Cash           {f(nc['Net Cash'], 'n'):<10}  {'POSITIVE' if nc['Positive'] else 'NEGATIVE'}")
        print(f"     Net Cash / Assets  {f(nc['Net Cash / Assets'])}")

        # 3. Cash Flow
        cf = r["cash_flow"]
        print(f"\n  3. OPERATING CASH FLOW ({cf['Score']}/100)")
        print(f"     OCF                {f(cf['Operating CF'], 'n'):<10}  Net Income        {f(cf['Net Income'], 'n')}")
        print(f"     OCF / Net Income   {f(cf['OCF / Net Income'], 'x'):<10}  Cumulative        {f(cf['Cumulative OCF/NI'], 'x')}")
        print(f"     Free Cash Flow     {f(cf['Free Cash Flow'], 'n'):<10}  FCF Margin        {f(cf['FCF Margin'])}")

        # 4. Capital Allocation
        ca = r["capital_allocation"]
        print(f"\n  4. CAPITAL ALLOCATION ({ca['Score']}/100)")
        print(f"     ROE                {f(ca['ROE']):<10}  ROIC              {f(ca['ROIC'])}")
        print(f"     ROA                {f(ca['ROA']):<10}  Payout Ratio      {f(ca['Payout Ratio'])}")
        print(f"     Dividends          {f(ca['Dividends'], 'n'):<10}  Buybacks          {f(ca['Buybacks'], 'n')}")

        # 5. Earnings Quality
        eq = r["earnings_quality"]
        print(f"\n  5. SUSTAINABLE EARNINGS ({eq['Score']}/100)")
        print(f"     Core Profit        {f(eq['Core Profit'], 'n'):<10}  Core Profit %     {f(eq['Core Profit %'])}")
        print(f"     Non-Recurring      {f(eq['Non-Recurring Items'], 'n'):<10}  OCF / NI          {f(eq['OCF / Net Income'], 'x')}")
        stab = eq.get("Margin Stability (CV)")
        if stab is not None:
            label = "Stable" if stab < 0.3 else "Moderate" if stab < 1.0 else "Volatile"
            print(f"     Margin Stability   {label} (CV={stab:.2f})")

        # 6. Tangible Book
        tb = r["tangible_book"]
        print(f"\n  6. TANGIBLE BOOK VALUE ({tb['Score']}/100)")
        print(f"     Total Equity       {f(tb['Total Equity'], 'n')}")
        print(f"     Goodwill           {f(tb['Goodwill'], 'n'):<10}  Goodwill/Equity   {f(tb['Goodwill / Equity'])}")
        print(f"     Intangibles        {f(tb['Intangible Assets'], 'n'):<10}  Intang/Equity     {f(tb['Intangibles / Equity'])}")
        print(f"     Tangible BV        {f(tb['Tangible Book Value'], 'n'):<10}  P/TBV             {f(tb['Price / Tangible BV'], 'x')}")

        # Summary
        if r.get("highlights"):
            print(f"\n  + {' / '.join(r['highlights'])}")
        if r.get("risks"):
            print(f"  - {' / '.join(r['risks'])}")

        # ── AI Analysis ──
        print(f"\n{'─' * W}")
        print("  ANALYSIS (AI-powered)")
        print(f"{'─' * W}")
        ai_text = self._ai_summary()
        print(ai_text)

        print(f"\n{'=' * W}")
        print("  Disclaimer: For reference only. Not investment advice.")
        print(f"{'=' * W}\n")

    def _build_data_prompt(self) -> str:
        """Build a structured data summary for the AI to analyze."""
        r = self.results
        d = self.latest
        name = self.ticker.upper()

        parts = [f"Company: {name}, FY{d.year}, Score: {r['total_score']}/100\n"]

        # 1
        lv = r["leverage"]
        parts.append(f"1. LEVERAGE (Score {lv['Score']}/100)")
        parts.append(f"   Debt/Assets={lv['Debt/Assets']:.1%}, Debt/Equity={lv['Debt/Equity']:.2f}x")
        ic_val = lv["Interest Coverage"]
        ic_str = "INF" if ic_val == float("inf") else f"{ic_val:.1f}x"
        parts.append(f"   Current Ratio={lv['Current Ratio']:.2f}x, Interest Coverage={ic_str}")
        parts.append(f"   Short-Term Debt %={lv['Short-Term Debt %']:.1%}")

        # 2
        nc = r["net_cash"]
        parts.append(f"\n2. NET CASH (Score {nc['Score']}/100)")
        parts.append(f"   Cash+ST Inv={nc['Cash + ST Investments']:,.0f}, Total Debt={nc['Total Debt']:,.0f}")
        parts.append(f"   Net Cash={nc['Net Cash']:,.0f} ({'Positive' if nc['Positive'] else 'Negative'})")
        trend = nc.get("Trend", [])
        if len(trend) >= 2:
            parts.append(f"   Trend: {trend[0]['year']}={trend[0]['net_cash']:,.0f} -> {trend[-1]['year']}={trend[-1]['net_cash']:,.0f}")

        # 3
        cf = r["cash_flow"]
        parts.append(f"\n3. OPERATING CASH FLOW (Score {cf['Score']}/100)")
        parts.append(f"   OCF={cf['Operating CF']:,.0f}, Net Income={cf['Net Income']:,.0f}")
        parts.append(f"   OCF/NI={cf['OCF / Net Income']:.2f}x, Cumulative OCF/NI={cf['Cumulative OCF/NI']:.2f}x")
        parts.append(f"   FCF={cf['Free Cash Flow']:,.0f}, FCF Margin={cf['FCF Margin']:.1%}")

        # 4
        ca = r["capital_allocation"]
        parts.append(f"\n4. CAPITAL ALLOCATION (Score {ca['Score']}/100)")
        parts.append(f"   ROE={ca['ROE']:.1%}, ROA={ca['ROA']:.1%}, ROIC={ca['ROIC']:.1%}")
        parts.append(f"   Dividends={ca['Dividends']:,.0f}, Buybacks={ca['Buybacks']:,.0f}")
        parts.append(f"   Payout Ratio={ca['Payout Ratio']:.1%}")

        # 5
        eq = r["earnings_quality"]
        parts.append(f"\n5. SUSTAINABLE EARNINGS (Score {eq['Score']}/100)")
        parts.append(f"   Core Profit={eq['Core Profit']:,.0f}, Core %={eq['Core Profit %']:.1%}")
        parts.append(f"   OCF/NI={eq['OCF / Net Income']:.2f}x")
        stab = eq.get("Margin Stability (CV)")
        if stab is not None:
            parts.append(f"   Margin Stability CV={stab:.2f}")

        # 6
        tb = r["tangible_book"]
        parts.append(f"\n6. TANGIBLE BOOK VALUE (Score {tb['Score']}/100)")
        parts.append(f"   Equity={tb['Total Equity']:,.0f}, Goodwill={tb['Goodwill']:,.0f}")
        parts.append(f"   Goodwill/Equity={tb['Goodwill / Equity']:.1%}")
        parts.append(f"   Tangible BV={tb['Tangible Book Value']:,.0f}")

        if r.get("highlights"):
            parts.append(f"\nHighlights: {', '.join(r['highlights'])}")
        if r.get("risks"):
            parts.append(f"Risks: {', '.join(r['risks'])}")

        return "\n".join(parts)

    def _ai_summary(self) -> str:
        """Call AI API (OpenAI or Anthropic) to generate analysis in Chinese."""
        data_text = self._build_data_prompt()
        name = self.ticker.upper()

        prompt = f"""You are a senior equity research analyst. Below is the 6-factor fundamental
analysis data for {name}. Write a concise but insightful analysis in Chinese (Mandarin).

Requirements:
- Use Chinese for the entire analysis
- Cover all 6 factors, explain what each number means for this specific company
- Consider industry context (what sector is {name} in, is the leverage normal for its industry?)
- Point out the most important 2-3 things an investor should focus on
- End with a clear bottom-line summary
- Keep it under 400 words
- Do NOT use markdown headers or bullet points, write in flowing paragraphs
- Be direct and opinionated, not generic

Data:
{data_text}"""

        # Try OpenAI first, then Anthropic, then fallback
        result = self._try_openai(prompt)
        if result:
            return result

        result = self._try_anthropic(prompt)
        if result:
            return result

        print("  [No AI API available, using rule-based analysis]")
        print("  [Set OPENAI_API_KEY or ANTHROPIC_API_KEY to enable AI]\n")
        return self._fallback_summary()

    def _try_openai(self, prompt: str) -> str | None:
        """Try OpenAI ChatGPT API."""
        import os
        if not os.environ.get("OPENAI_API_KEY"):
            return None
        try:
            from openai import OpenAI
        except ImportError:
            try:
                import subprocess, sys
                subprocess.check_call([sys.executable, "-m", "pip", "install",
                                       "openai", "--break-system-packages", "-q"])
                from openai import OpenAI
            except Exception:
                return None
        try:
            client = OpenAI()
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                max_tokens=1500,
                messages=[{"role": "user", "content": prompt}]
            )
            text = response.choices[0].message.content
            return "  " + text.replace("\n", "\n  ")
        except Exception as e:
            print(f"  [OpenAI error: {e}]")
            return None

    def _try_anthropic(self, prompt: str) -> str | None:
        """Try Anthropic Claude API."""
        import os
        if not os.environ.get("ANTHROPIC_API_KEY"):
            return None
        try:
            import anthropic
        except ImportError:
            try:
                import subprocess, sys
                subprocess.check_call([sys.executable, "-m", "pip", "install",
                                       "anthropic", "--break-system-packages", "-q"])
                import anthropic
            except Exception:
                return None
        try:
            client = anthropic.Anthropic()
            message = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=1500,
                messages=[{"role": "user", "content": prompt}]
            )
            return "  " + message.content[0].text.replace("\n", "\n  ")
        except Exception as e:
            print(f"  [Anthropic error: {e}]")
            return None

    def _fallback_summary(self) -> str:
        """Rule-based fallback when AI is not available."""
        r = self.results
        d = self.latest
        name = self.ticker.upper()
        lines = []

        lv = r["leverage"]
        dar = lv["Debt/Assets"]
        ic = lv["Interest Coverage"]
        if dar < 0.3:
            lines.append(f"  {name} 的资产负债率仅 {dar:.0%}，负债水平很低，财务结构保守稳健。")
        elif dar < 0.5:
            lines.append(f"  {name} 的资产负债率为 {dar:.0%}，负债处于中等水平，整体可控。")
        else:
            lines.append(f"  {name} 的资产负债率高达 {dar:.0%}，杠杆较重，一旦经营恶化，偿债压力会很大。")
        if ic != float("inf") and ic < 3:
            lines.append(f"  利息覆盖倍数仅 {ic:.1f}x，偏低——如果利润下滑，可能连利息都还不起。")

        nc = r["net_cash"]
        net = nc["Net Cash"]
        if nc["Positive"]:
            lines.append(f"\n  公司净现金为 {net:,.0f}(正数)，把所有借款还清后手上还有余钱。")
        else:
            lines.append(f"\n  净现金为 {net:,.0f}(负数)，经营依赖借债，安全边际较低。")

        cf = r["cash_flow"]
        ocf_ni = cf["OCF / Net Income"]
        if ocf_ni > 1.0:
            lines.append(f"\n  现金流质量好：每赚 1 块利润，实际收到 {ocf_ni:.2f} 块现金。")
        elif ocf_ni < 0.6:
            lines.append(f"\n  注意：经营现金流只有净利润的 {ocf_ni:.2f} 倍，利润含水分。")

        ca = r["capital_allocation"]
        roic = ca["ROIC"]
        if roic > 0.15:
            lines.append(f"\n  ROIC 达到 {roic:.1%}，资金使用效率高，可能有竞争壁垒。")
        elif roic < 0.08:
            lines.append(f"\n  ROIC 仅 {roic:.1%}，投入资本回报偏低。")

        eq = r["earnings_quality"]
        cp = eq["Core Profit %"]
        if cp > 0.9:
            lines.append(f"\n  {cp:.0%} 的利润来自主营业务，盈利质量高。")
        elif cp < 0.6:
            lines.append(f"\n  只有 {cp:.0%} 来自主营，其余是一次性收益，盈利质量堪忧。")

        tb = r["tangible_book"]
        gw_eq = tb["Goodwill / Equity"]
        if gw_eq > 0.4:
            lines.append(f"\n  商誉占净资产 {gw_eq:.0%}，比例过重，需警惕减值风险。")
        elif gw_eq < 0.05:
            lines.append(f"\n  商誉极低({gw_eq:.1%})，资产很扎实。")

        ts = r["total_score"]
        lines.append(f"\n  总结：{name} 综合得分 {ts}/100。")
        if ts >= 75:
            lines.append(f"  各项基本面表现扎实，值得进一步深入研究。")
        elif ts >= 55:
            lines.append(f"  基本面有亮点也有短板，务必搞清楚薄弱环节背后的原因。")
        else:
            lines.append(f"  多个维度亮红灯，需要非常谨慎。")

        return "\n".join(lines)

    # ── Export ──
    def to_json(self, filepath: str = "report.json"):
        if not self.results: self.run_full_analysis()
        def conv(o):
            if isinstance(o, float) and (math.isinf(o) or math.isnan(o)): return None
            if isinstance(o, (np.floating, np.integer)): return float(o)
            return o
        with open(filepath, "w") as f:
            json.dump(self.results, f, indent=2, default=conv)
        print(f"  Exported: {filepath}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Data Fetcher (yfinance)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def fetch(ticker: str, years: int = 5):
    """Fetch annual data from Yahoo Finance."""
    try:
        import yfinance as yf
    except ImportError:
        print("Install yfinance first: pip install yfinance")
        return []

    stock = yf.Ticker(ticker)
    info = stock.info
    inc = stock.financials.T
    bal = stock.balance_sheet.T
    cf = stock.cashflow.T

    market_cap = info.get("marketCap", 0) or 0
    shares = info.get("sharesOutstanding", 0) or 0
    price = info.get("currentPrice", 0) or info.get("regularMarketPrice", 0) or 0

    data_list = []
    for idx in inc.index[:years]:
        year = idx.year

        def get(df, *keys, default=0.0):
            for k in keys:
                if k in df.columns:
                    val = df.loc[idx, k]
                    if pd.notna(val): return float(val)
            return default

        fd = FinancialData(year=year)
        fd.revenue = get(inc, "Total Revenue", "Revenue")
        fd.cost_of_revenue = get(inc, "Cost Of Revenue")
        fd.gross_profit = get(inc, "Gross Profit")
        fd.operating_income = get(inc, "Operating Income", "EBIT")
        fd.net_income = get(inc, "Net Income", "Net Income Common Stockholders")
        fd.ebitda = get(inc, "EBITDA", "Normalized EBITDA")
        fd.interest_expense = abs(get(inc, "Interest Expense"))

        if idx in bal.index:
            fd.total_assets = get(bal, "Total Assets")
            fd.total_liabilities = get(bal, "Total Liabilities Net Minority Interest", "Total Liab")
            fd.total_equity = get(bal, "Stockholders Equity", "Total Stockholder Equity", "Common Stock Equity")
            fd.goodwill = get(bal, "Goodwill")
            fd.intangible_assets = get(bal, "Intangible Assets", "Other Intangible Assets")
            fd.cash_and_equivalents = get(bal, "Cash And Cash Equivalents", "Cash")
            fd.short_term_investments = get(bal, "Short Term Investments", "Other Short Term Investments")
            fd.total_debt = get(bal, "Total Debt")
            fd.short_term_debt = get(bal, "Short Long Term Debt", "Current Debt")
            fd.long_term_debt = get(bal, "Long Term Debt")
            fd.current_assets = get(bal, "Current Assets", "Total Current Assets")
            fd.current_liabilities = get(bal, "Current Liabilities", "Total Current Liabilities")
            fd.fixed_assets = get(bal, "Net PPE", "Property Plant Equipment")

        if idx in cf.index:
            fd.operating_cash_flow = get(cf, "Total Cash From Operating Activities", "Operating Cash Flow")
            fd.capex = abs(get(cf, "Capital Expenditures", "Capital Expenditure"))
            fd.free_cash_flow = get(cf, "Free Cash Flow")
            if fd.free_cash_flow == 0 and fd.operating_cash_flow:
                fd.free_cash_flow = fd.operating_cash_flow - fd.capex
            fd.dividends_paid = abs(get(cf, "Dividends Paid", "Common Stock Dividend Paid"))
            fd.share_buyback = abs(get(cf, "Repurchase Of Stock", "Common Stock Repurchased"))

        fd.market_cap = market_cap
        fd.shares_outstanding = shares
        fd.stock_price = price
        data_list.append(fd)

    return data_list


def analyze(ticker: str, years: int = 5):
    """One-liner: fetch + analyze + print."""
    print(f"\n  Fetching {ticker.upper()}...")
    data = fetch(ticker, years)
    if not data:
        print(f"  No data found for {ticker}")
        return None
    a = SixFactorAnalyzer(data, ticker)
    a.run_full_analysis()
    a.print_report()
    return a


# ═══════════════════════════════════════════════════════
#  >>> EDIT HERE <<<
# ═══════════════════════════════════════════════════════

# Step 1: Paste your API key (pick one)
import os
os.environ["OPENAI_API_KEY"] = ""      # <-- Paste OpenAI key here, e.g. "sk-xxx..."
os.environ["ANTHROPIC_API_KEY"] = ""   # <-- Or paste Anthropic key here, e.g. "sk-ant-xxx..."

# Step 2: Change ticker(s)
tickers = ["SNDK"]  # <-- Change here. e.g. ["AAPL", "MSFT", "0700.HK"]

for t in tickers:
    analyze(t)